In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests
import os
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.append(os.path.abspath(".."))
import src.raw_preprocessing as rp
import src.feature_engineering as fe
import src.create_dataset as cd
from datetime import date, datetime, timedelta
from vacances_scolaires_france import SchoolHolidayDates

import openmeteo_requests
import requests_cache
from retry_requests import retry

In [244]:
conso = pd.read_parquet("../data/final_datasets/datasets_linear_models/conso_v3_linear.parquet")

In [353]:
conso.columns

Index(['Consommation', 'Zone_A', 'Zone_B', 'Zone_C',
       'Vacances de la Toussaint', 'Vacances de Noël', 'Vacances d'Hiver',
       'Vacances de Printemps', 'Vacances d'Été', 'public_holidays', '44T',
       '69T', '59T', '75T', '13T', '33T', 'T', 'U', 'FF', 'PMER', 'RR1',
       'year', 'month', 'hour', 'day_of_week', 'is_weekend', 'hour_sin',
       'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin',
       'month_cos', 'lagged_1', 'lagged_2', 'lagged_48', 'lagged_336',
       'rolling_mean_24h', 'rolling_std_24h', 'rolling_mean_7d',
       'rolling_std_7d', 'rolling_max_24h', 'rolling_min_24h',
       'consumption_diff_1', 'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter', 'temp_sq', 'humidity_sq', 'hour_x_is_weekend',
       'hour_x_is_holiday', 'hour_x_dow', 'hour_x_month', 'is_weekend_x_month',
       'is_holiday_x_month', 'hour_x_temp', 'hour_x_humidity', 'hour_x_wind',
  

In [166]:
cd.download_monthly_data()

conso_energie_2026.zip already exists


In [321]:
df = rp.conso_preprocess(Path("../data/conso/real_time_conso/"))

df = df[df["Heures"].apply(lambda x: x.minute in {00, 30})]

df = df.reset_index(drop=True)
df.loc[len(df)] = None

In [322]:
df.columns

Index(['Date', 'Heures', 'Consommation'], dtype='object')

In [323]:
df = fe.lagged_consumption(df)

In [324]:
t = df["Heures"].iloc[-2]

new_time = (
    datetime.combine(datetime.today(), t)
    + timedelta(minutes=30)
).time()

In [325]:
df.loc[len(df)-1, "Heures"] = new_time

In [326]:
df.loc[len(df)-1, "Date"] = datetime.today().strftime('%Y-%m-%d')

In [327]:
df

,Date,Heures,Consommation,lagged_1,lagged_2,lagged_48,lagged_336
0,2026-05-01,00:00:00,41688.0,NaN,NaN,NaN,NaN
1,2026-05-01,00:30:00,40393.0,41688.0,NaN,NaN,NaN
2,2026-05-01,01:00:00,38108.0,40393.0,41688.0,NaN,NaN
3,2026-05-01,01:30:00,37744.0,38108.0,40393.0,NaN,NaN
4,2026-05-01,02:00:00,36708.0,37744.0,38108.0,NaN,NaN
...,...,...,...,...,...,...,...
3818,2026-07-19,13:00:00,42182.0,42953.0,41863.0,46948.0,45512.0
3819,2026-07-19,13:30:00,41187.0,42182.0,42953.0,46739.0,44682.0
3820,2026-07-19,14:00:00,41283.0,41187.0,42182.0,46889.0,44856.0
3821,2026-07-19,14:30:00,41408.0,41283.0,41187.0,46745.0,45452.0


#### Adding the name of the holidays

In [328]:
from pprint import pprint
today = date(2026, 5, 16).isoformat()

url = "https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/fr-en-calendrier-scolaire/records"

params = {
    "where": f"start_date <= date'{today}' AND end_date >= date'{today}' AND zones = 'Zone C'"
}

response = requests.get(url, params=params)

In [329]:
#Adding the infos about the holidays
today = date.today().isoformat()
url = "https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/fr-en-calendrier-scolaire/records"

zones = ['A', 'B', 'C']
vacances = ['vacances de la toussaint', 'vacances de noël', "vacances d'hiver",'vacances de printemps', "vacances d'été"]
feries = [
    "jour de l'an",
    "lundi de pâques",
    "fête du travail",
    "victoire 1945",
    "ascension",
    "lundi de pentecôte",
    "fête nationale",
    "assomption",
    "toussaint",
    "armistice",
    "noël",
    "pont de l'ascension",
]

#finished_with_holidays = False
df.loc[:, "Zone_A"] = 0
df.loc[:, "Zone_B"] = 0
df.loc[:, "Zone_C"] = 0

df.loc[:, "public_holidays"] = 0
df.loc[:, "Vacances de la Toussaint"] = 0
df.loc[:, "Vacances de Noël"] = 0
df.loc[:, "Vacances d'Hiver"] = 0
df.loc[:, "Vacances de Printemps"] = 0
df.loc[:, "Vacances d'Été"] = 0

for z in zones:
    params = {
        "where": f"start_date <= date'{today}' AND end_date >= date'{today}' AND zones = 'Zone {z}'"
    }
    response = requests.get(url, params=params).json()
    # If no holidays we set the columns with the value 0
    if response["total_count"] == 0:
        continue

    else : 
        
        for event in response["results"]:
            # We check if it's school holidays
            if event["description"].lower() in vacances:
                df.loc[:, f"Zone_{z}"] = 1
                if event["description"].lower() == "vacances de la toussaint":
                    df.loc[:, "Vacances de la Toussaint"] = 1
                    
                elif event["description"].lower() == "vacances de noël":
                    df.loc[:, "Vacances de Noël"] = 1
                    
                elif event["description"].lower() == "vacances d'hiver":
                    df.loc[:, "Vacances d'Hiver"] = 1
                    
                elif event["description"].lower() == "vacances de printemps":
                    df.loc[:, "Vacances de Printemps"] = 1

                elif event["description"].lower() == "vacances d'été":
                    df.loc[:, "Vacances d'Été"] = 1
            # Or if it's public holidays (jours fériés)
            elif event in feries:
                df.loc[:, "public_holidays"] = 1

In [330]:
df

,Date,Heures,Consommation,lagged_1,lagged_2,lagged_48,lagged_336,Zone_A,Zone_B,Zone_C,public_holidays,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été
0,2026-05-01,00:00:00,41688.0,NaN,NaN,NaN,NaN,1,1,1,0,0,0,0,0,1
1,2026-05-01,00:30:00,40393.0,41688.0,NaN,NaN,NaN,1,1,1,0,0,0,0,0,1
2,2026-05-01,01:00:00,38108.0,40393.0,41688.0,NaN,NaN,1,1,1,0,0,0,0,0,1
3,2026-05-01,01:30:00,37744.0,38108.0,40393.0,NaN,NaN,1,1,1,0,0,0,0,0,1
4,2026-05-01,02:00:00,36708.0,37744.0,38108.0,NaN,NaN,1,1,1,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3818,2026-07-19,13:00:00,42182.0,42953.0,41863.0,46948.0,45512.0,1,1,1,0,0,0,0,0,1
3819,2026-07-19,13:30:00,41187.0,42182.0,42953.0,46739.0,44682.0,1,1,1,0,0,0,0,0,1
3820,2026-07-19,14:00:00,41283.0,41187.0,42182.0,46889.0,44856.0,1,1,1,0,0,0,0,0,1
3821,2026-07-19,14:30:00,41408.0,41283.0,41187.0,46745.0,45452.0,1,1,1,0,0,0,0,0,1


In [331]:
df = fe.date_and_hour_pred(df)
df = fe.cyclical_encoding(df)
df = fe.rolling_window(df)
df = fe.lagged_trend(df)
df = fe.seasons_linear(df)
df = df.drop(["Consommation"], axis=1)

In [332]:
df.columns

Index(['lagged_1', 'lagged_2', 'lagged_48', 'lagged_336', 'Zone_A', 'Zone_B',
       'Zone_C', 'public_holidays', 'Vacances de la Toussaint',
       'Vacances de Noël', 'Vacances d'Hiver', 'Vacances de Printemps',
       'Vacances d'Été', 'full_date', 'year', 'month', 'hour', 'day_of_week',
       'is_weekend', 'hour_sin', 'hour_cos', 'day_of_week_sin',
       'day_of_week_cos', 'month_sin', 'month_cos', 'rolling_mean_24h',
       'rolling_std_24h', 'rolling_mean_7d', 'rolling_std_7d',
       'rolling_max_24h', 'rolling_min_24h', 'consumption_diff_1',
       'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter'],
      dtype='object')

In [334]:
row = df.iloc[-1]

In [341]:
pred = pd.DataFrame([row] * 10)

In [342]:
pred["full_date"] = row["full_date"] + pd.to_timedelta(range(10), unit="m") * 30
pred = pred.reset_index(drop=True)

In [346]:
pred.columns

Index(['lagged_1', 'lagged_2', 'lagged_48', 'lagged_336', 'Zone_A', 'Zone_B',
       'Zone_C', 'public_holidays', 'Vacances de la Toussaint',
       'Vacances de Noël', 'Vacances d'Hiver', 'Vacances de Printemps',
       'Vacances d'Été', 'full_date', 'year', 'month', 'hour', 'day_of_week',
       'is_weekend', 'hour_sin', 'hour_cos', 'day_of_week_sin',
       'day_of_week_cos', 'month_sin', 'month_cos', 'rolling_mean_24h',
       'rolling_std_24h', 'rolling_mean_7d', 'rolling_std_7d',
       'rolling_max_24h', 'rolling_min_24h', 'consumption_diff_1',
       'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter'],
      dtype='object')

### RTE France API

In [86]:
id_client = "863fa354-33b4-4bf6-a844-3dd668062f92"
id_secret = "83b2284d-b40f-4627-916f-d35473030cbf"
url = "https://digital.iservices.rte-france.com/token/oauth/"

response = requests.post(url, auth=(id_client, id_secret))

In [87]:
response.json()

{'access_token': 'k474YPtYGg8Im6fzkvcoIUeWaPI4fzxRjj6ankSZK5EghTijE48cHG',
 'token_type': 'Bearer',
 'expires_in': 3600}

In [88]:
token = response.json()["access_token"]
headers = {
    "Authorization" : f"Bearer {token}"
}

url = "https://digital.iservices.rte-france.com/open_api/consumption/v1/short_term"

data = requests.get(url, headers=headers)

In [89]:
data.json()

{'short_term': [{'type': 'REALISED',
   'start_date': '2026-07-17T00:00:00+02:00',
   'end_date': '2026-07-18T00:00:00+02:00',
   'values': [{'start_date': '2026-07-17T00:00:00+02:00',
     'end_date': '2026-07-17T00:15:00+02:00',
     'updated_date': '2026-07-17T13:05:47+02:00',
     'value': 47387},
    {'start_date': '2026-07-17T00:15:00+02:00',
     'end_date': '2026-07-17T00:30:00+02:00',
     'updated_date': '2026-07-17T13:05:47+02:00',
     'value': 46988},
    {'start_date': '2026-07-17T00:30:00+02:00',
     'end_date': '2026-07-17T00:45:00+02:00',
     'updated_date': '2026-07-17T13:05:48+02:00',
     'value': 45765},
    {'start_date': '2026-07-17T00:45:00+02:00',
     'end_date': '2026-07-17T01:00:00+02:00',
     'updated_date': '2026-07-17T13:05:48+02:00',
     'value': 44762},
    {'start_date': '2026-07-17T01:00:00+02:00',
     'end_date': '2026-07-17T01:15:00+02:00',
     'updated_date': '2026-07-17T13:05:49+02:00',
     'value': 43709},
    {'start_date': '2026-07-17T01

### Open-Meteo API

In [347]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
lat_long = {
    '13': {'ville': 'Marseille', 'latitude': 43.2965, 'longitude': 5.3698},     # Bouches-du-Rhône
    '33': {'ville': 'Bordeaux',  'latitude': 44.8378, 'longitude': -0.5792},    # Gironde
    '44': {'ville': 'Nantes',    'latitude': 47.2184, 'longitude': -1.5536},    # Loire-Atlantique
    '59': {'ville': 'Lille',     'latitude': 50.6292, 'longitude': 3.0573},     # Nord
    '69': {'ville': 'Lyon',      'latitude': 45.7640, 'longitude': 4.8357},     # Rhône
    '75': {'ville': 'Paris',     'latitude': 48.8566, 'longitude': 2.3522},     # Paris
}

station_population = {
    '13': 2087658,   # Bouches-du-Rhône 
    '33': 1690493,   # Gironde           
    '44': 1487570,   # Loire-Atlantique   
    '59': 2615635,   # Nord              
    '69': 1914667,   # Rhône             
    '75': 2103778,   # Paris
}

total_pop = sum(station_population.values())
weights = {city: pop / total_pop for city, pop in station_population.items()}


cols = ['T', 'U', 'FF', 'PMER', 'RR1']
df_temp = pd.DataFrame(
    np.zeros(shape=(pred.shape[0], len(cols))),
    columns=cols
)

for k, v in lat_long.items():

    # Calling the API for our department
    params = {
    	"latitude": v["latitude"],
    	"longitude": v["longitude"],
    	"hourly": ["temperature_2m", "relative_humidity_2m", "rain", "surface_pressure", "wind_speed_10m"],
    	"timezone": "Europe/London",
    	"past_days": 7,
    	"forecast_days": 1,
    }
    responses = openmeteo.weather_api(url, params = params)
    
    # Process first location. Add a for-loop for multiple locations or weather models
    response = responses[0]
    print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation: {response.Elevation()} m asl")
    print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
    print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
    
    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_rain = hourly.Variables(2).ValuesAsNumpy()
    hourly_surface_pressure = hourly.Variables(3).ValuesAsNumpy()
    hourly_wind_speed_10m = hourly.Variables(4).ValuesAsNumpy()
    
    hourly_data = {
    	"date": pd.date_range(
    		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
    		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
    		freq = pd.Timedelta(seconds = hourly.Interval()),
    		inclusive = "left"
    	).tz_convert(response.Timezone().decode())
    }

    # Preprocessing the weather date
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["rain"] = hourly_rain
    hourly_data["surface_pressure"] = hourly_surface_pressure
    hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
    
    hourly_dataframe = pd.DataFrame(data = hourly_data)

    hourly_dataframe = hourly_dataframe.rename(columns={
        "temperature_2m" : "T",
        "relative_humidity_2m": "U",
        "rain" : "RR1",
        "surface_pressure" : "PMER",
        "wind_speed_10m" : "FF"
    })

    # We add the 30min weather date 
    hourly_dataframe["date"] = pd.to_datetime(hourly_dataframe["date"])
    hourly_dataframe = hourly_dataframe.set_index("date")
    hourly_dataframe = hourly_dataframe.asfreq("30min")

    # We interpolate the missing values
    hourly_dataframe["RR1"] = hourly_dataframe["RR1"].fillna(0)
    hourly_dataframe = hourly_dataframe.interpolate(method="time")
    hourly_dataframe = hourly_dataframe.reset_index()
    hourly_dataframe["date"] = hourly_dataframe["date"].dt.tz_localize(None)
    hourly_dataframe = hourly_dataframe[hourly_dataframe["date"].isin(pred["full_date"])]
    hourly_dataframe = hourly_dataframe.reset_index()
    
    print(hourly_dataframe)
    pred[f"{k}T"] = hourly_dataframe[['T']]


    population = weights[k]
    df_temp += hourly_dataframe[cols].values * population


pred = pd.concat([pred, df_temp], axis=1)

Coordinates: 43.29999923706055°N 5.369998931884766°E
Elevation: 6.0 m asl
Timezone: b'Europe/London'b'GMT+1'
Timezone difference to GMT+0: 3600s
   index                date          T      U  RR1         PMER         FF
0    366 2026-08-01 15:00:00  28.017000  67.00  0.0  1011.212891  28.483257
1    367 2026-08-01 15:30:00  27.917000  66.00  0.0  1011.162598  26.285498
2    368 2026-08-01 16:00:00  27.817001  65.00  0.0  1011.112366  24.087738
3    369 2026-08-01 16:30:00  27.817001  64.50  0.0  1010.962341  20.949776
4    370 2026-08-01 17:00:00  27.817001  64.00  0.0  1010.812317  17.811815
5    371 2026-08-01 17:30:00  27.817001  62.50  0.0  1010.662231  16.327497
6    372 2026-08-01 18:00:00  27.817001  61.00  0.0  1010.512146  14.843180
7    373 2026-08-01 18:30:00  27.773251  60.00  0.0  1010.661865  12.407305
8    374 2026-08-01 19:00:00  27.729500  59.00  0.0  1010.811584   9.971431
9    375 2026-08-01 19:30:00  27.473251  59.75  0.0  1011.060913  10.845100
Coordinates: 44.840

In [348]:
print(pred.columns)
pred

Index(['lagged_1', 'lagged_2', 'lagged_48', 'lagged_336', 'Zone_A', 'Zone_B',
       'Zone_C', 'public_holidays', 'Vacances de la Toussaint',
       'Vacances de Noël', 'Vacances d'Hiver', 'Vacances de Printemps',
       'Vacances d'Été', 'full_date', 'year', 'month', 'hour', 'day_of_week',
       'is_weekend', 'hour_sin', 'hour_cos', 'day_of_week_sin',
       'day_of_week_cos', 'month_sin', 'month_cos', 'rolling_mean_24h',
       'rolling_std_24h', 'rolling_mean_7d', 'rolling_std_7d',
       'rolling_max_24h', 'rolling_min_24h', 'consumption_diff_1',
       'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter', '13T', '33T', '44T', '59T', '69T', '75T', 'T', 'U',
       'FF', 'PMER', 'RR1'],
      dtype='object')


,lagged_1,lagged_2,lagged_48,lagged_336,Zone_A,Zone_B,Zone_C,public_holidays,Vacances de la Toussaint,Vacances de Noël,...,33T,44T,59T,69T,75T,T,U,FF,PMER,RR1
0,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,29.845499,27.737000,27.480499,30.911001,28.615000,28.695193,36.504272,10.924157,1010.888550,0.0
1,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,30.170500,27.937000,27.530499,31.261002,28.690001,28.829385,36.493680,11.015824,1010.630836,0.0
2,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,30.495501,28.136999,27.580500,31.611000,28.765001,28.963577,36.483087,11.107490,1010.373138,0.0
3,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,30.570499,28.261999,27.430500,31.810999,28.815001,28.997906,36.154021,10.815239,1010.215767,0.0
4,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,30.645500,28.386999,27.280499,32.010998,28.865000,29.032235,35.824954,10.522987,1010.058380,0.0
5,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,30.520500,28.336998,27.080498,31.885998,28.815001,28.935314,35.305140,10.347065,1009.895599,0.0
6,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,30.395500,28.286999,26.880499,31.761000,28.765001,28.838393,34.785325,10.171143,1009.732826,0.0
7,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,30.220501,28.118250,26.480499,31.410999,28.540001,28.600748,35.256625,9.774243,1009.715546,0.0
8,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,30.045500,27.949501,26.080500,31.061001,28.315001,28.363102,35.727926,9.377344,1009.698273,0.0
9,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,29.745499,27.555750,24.480499,30.636002,27.790001,27.713421,37.524247,11.200428,1009.804352,0.0


In [349]:
pred = fe.interactions_linear(pred)

In [354]:
print(len(pred.columns))
pred

83


,lagged_1,lagged_2,lagged_48,lagged_336,Zone_A,Zone_B,Zone_C,public_holidays,Vacances de la Toussaint,Vacances de Noël,...,is_weekend_x_season_Summer,is_holiday_x_season_Summer,hour_x_season_Winter,is_weekend_x_season_Winter,is_holiday_x_season_Winter,temp_x_humidity,temp_x_wind,humidity_x_wind,HDD,CDD
0,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,1,0,0.0,0,0,1047.497121,313.470794,398.778407,0.0,10.695193
1,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,1,0,0.0,0,0,1052.090351,317.579430,402.007959,0.0,10.829385
2,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,1,0,0.0,0,0,1056.680711,321.712657,405.235540,0.0,10.963577
3,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,1,0,0.0,0,0,1048.390887,313.619276,391.014370,0.0,10.997906
4,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,1,0,0.0,0,0,1040.078490,305.505823,376.985513,0.0,11.032235
5,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,1,0,0.0,0,0,1021.565288,299.395574,365.304577,0.0,10.935314
6,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,1,0,0.0,0,0,1003.152875,293.319426,353.806519,0.0,10.838393
7,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,1,0,0.0,0,0,1008.365849,279.550674,344.606830,0.0,10.600748
8,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,1,0,0.0,0,0,1013.354817,265.970570,335.033057,0.0,10.363102
9,41408.0,41283.0,46359.0,44998.0,1,1,1,0,0,0,...,1,0,0.0,0,0,1039.925259,310.402184,420.287628,0.0,9.713421
